# Zone Segmentation — End-to-End Demo

Interactive walkthrough of the **pattern-conditioned U-Net** in `model/zone_segmentation/`.

Covers:
1. Loading the synthetic zoning dataset
2. Visualizing `(image, pattern, mask)` triplets
3. Building the model and inspecting its architecture
4. Running a short training loop with the upgraded `Trainer`
5. Plotting training curves (loss / IoU / Dice / LR)
6. Visualizing predictions vs. ground truth

**Optional:** if `WANDB_API_KEY` is set in your environment, the trainer will automatically
log metrics, sample predictions, and the final model artifact to Weights & Biases — same
pattern used in `ac215_Spatially`.

## 1. Setup

In [ ]:
import sys, pathlib

# Make the repo root importable so `model.zone_segmentation` works.
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repo root:', REPO_ROOT)

In [ ]:
import logging
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('Using device:', DEVICE)
print('Torch:', torch.__version__)

## 2. Load the dataset

The synthetic dataset lives at `data/training/zoning_segmentation/`. Each sample contains a
rendered map (`images/<id>.png`), per-zone binary masks (`masks/<id>/zone_XX/mask.png`), and
32×32 pattern thumbnails (`masks/<id>/zone_XX/pattern.png`).

In [ ]:
from model.zone_segmentation.dataset import ZoneSegmentationDataset, get_dataloaders

DATA_ROOT = REPO_ROOT / 'data' / 'training' / 'zoning_segmentation'
IMAGE_SIZE = 256  # smaller than 512 so the notebook trains fast on CPU/MPS
BATCH_SIZE = 2

train_loader, val_loader = get_dataloaders(
    root=str(DATA_ROOT),
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    num_workers=0,
)
print(f'Train pairs: {len(train_loader.dataset)}')
print(f'Val pairs:   {len(val_loader.dataset)}')
print(f'Train batches: {len(train_loader)}, val batches: {len(val_loader)}')

### Inspect a single sample

In [ ]:
image, pattern, mask = train_loader.dataset[0]
print('image  :', tuple(image.shape), image.dtype)
print('pattern:', tuple(pattern.shape), pattern.dtype)
print('mask   :', tuple(mask.shape), mask.dtype, '| unique:', torch.unique(mask).tolist())

### Visualize a few triplets

In [ ]:
import matplotlib.pyplot as plt
from model.zone_segmentation.viz import plot_dataset_samples

fig = plot_dataset_samples(train_loader.dataset, n=4, seed=0)
plt.show()

## 3. Build the model

`PatternConditionedUNet` = ResNet-34 encoder + FiLM-conditioned U-Net decoder. The pattern
thumbnail is encoded to a 256-dim vector that gates each decoder block.

In [ ]:
from model.zone_segmentation.unet import PatternConditionedUNet

model = PatternConditionedUNet(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Total params:    {n_params:.2f} M')
print(f'Trainable params:{n_trainable:.2f} M')

In [ ]:
# Forward-pass shape check
with torch.no_grad():
    img = image.unsqueeze(0).to(DEVICE)
    pat = pattern.unsqueeze(0).to(DEVICE)
    out = model(img, pat)
print('input  :', tuple(img.shape))
print('pattern:', tuple(pat.shape))
print('output :', tuple(out.shape))

### Module breakdown

In [ ]:
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters()) / 1e6
    print(f'  {name:18s} {n:6.2f} M params  ({type(module).__name__})')

## 4. Train

The upgraded `Trainer` tracks:
- per-epoch train/val loss, IoU, Dice, F1
- learning-rate schedule
- mixed-precision (auto on CUDA)
- gradient clipping
- prediction snapshots saved to `checkpoints/.../samples/epoch_XXX.png`
- a `history.json` we can replot any time
- (optional) Weights & Biases run if `WANDB_API_KEY` is set

It returns a `history` dict that we plot below.

In [ ]:
from model.zone_segmentation.trainer import Trainer

EPOCHS = 5  # bump this for real training; 5 is enough to see curves move
SAVE_DIR = REPO_ROOT / 'checkpoints' / 'zone_segmentation_notebook'

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-4,
    epochs=EPOCHS,
    device=DEVICE,
    save_dir=str(SAVE_DIR),
    sample_log_every=1,  # snapshot every epoch in this short demo
    wandb_config={'image_size': IMAGE_SIZE, 'notebook': True},
)
history = trainer.train()

## 5. Plot training history

In [ ]:
from model.zone_segmentation.viz import plot_history

fig = plot_history(history, save_path=SAVE_DIR / 'history.png')
plt.show()

You can also reload `history.json` later (e.g. after a long training run) and replot:
```python
import json
history = json.loads((SAVE_DIR / 'history.json').read_text())
plot_history(history)
```

## 6. Visualize predictions

In [ ]:
from model.zone_segmentation.viz import make_prediction_grid

fig = make_prediction_grid(model, val_loader, device=DEVICE, n=4)
plt.show()

### Where the snapshots live
Per-epoch prediction grids are also written to:
```
checkpoints/zone_segmentation_notebook/samples/epoch_001.png
checkpoints/zone_segmentation_notebook/samples/epoch_002.png
...
```
Useful for stitching into a training-progress GIF.

## 7. (Optional) Weights & Biases

If you have an account, set the API key before importing the trainer:
```bash
export WANDB_API_KEY=<your key>
```
Then re-run section 4 — the trainer will auto-detect it, create a run named
`unet-film-eN-bsB-<timestamp>`, log metrics under `train/*` and `val/*`, push the
prediction grid as a `wandb.Image`, and save the best checkpoint as a
`zone-segmentation-model` artifact.

This mirrors how `ac215_Spatially` tracks its NER training runs.